In [3]:
from torch.utils.data import random_split

train_dataset_full = datasets.ImageFolder(
    os.path.join(data_dir, "train"),
    transform=train_tf
)

test_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "test"),
    transform=val_tf
)

class_names = train_dataset_full.classes

print("Classes:", class_names)

train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

NameError: name 'datasets' is not defined

In [2]:
data_dir="/kaggle/input/datasets/freshwater-fish-dataset/Freshwater Fish Disease Aquaculture in south asia"

In [4]:
import os

for root, dirs, files in os.walk(data_dir):
    print(root)
    break

/kaggle/input/datasets/freshwater-fish-dataset/Freshwater Fish Disease Aquaculture in south asia


In [8]:
from torchvision import datasets
from torch.utils.data import random_split
import torch

full_dataset = datasets.ImageFolder(
    data_dir,
    transform=train_tf
)

class_names = full_dataset.classes

print("Classes:", class_names)

print("Total Images:", len(full_dataset))

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train Size:", len(train_dataset))
print("Validation Size:", len(val_dataset))

Classes: ['Test', 'Train']
Total Images: 2450
Train Size: 1960
Validation Size: 490


In [7]:
from torchvision import transforms

IMG_SIZE = 224
BATCH_SIZE = 16

train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [9]:
full_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "Train"),
    transform=train_tf
)

class_names = full_dataset.classes

print(class_names)

['Bacterial Red disease', 'Bacterial diseases - Aeromoniasis', 'Bacterial gill disease', 'Fungal diseases Saprolegniasis', 'Healthy Fish', 'Parasitic diseases', 'Viral diseases White tail disease']


In [10]:
from torchvision import datasets
from torch.utils.data import random_split
import torch
import os

full_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "Train"),
    transform=train_tf
)

class_names = full_dataset.classes

print("Classes:", class_names)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

test_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "Test"),
    transform=val_tf
)

print("Train Size:", len(train_dataset))
print("Validation Size:", len(val_dataset))
print("Test Size:", len(test_dataset))

Classes: ['Bacterial Red disease', 'Bacterial diseases - Aeromoniasis', 'Bacterial gill disease', 'Fungal diseases Saprolegniasis', 'Healthy Fish', 'Parasitic diseases', 'Viral diseases White tail disease']
Train Size: 1400
Validation Size: 350
Test Size: 700


In [13]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models

from torch.utils.data import (
    DataLoader,
    random_split
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

In [14]:
print(class_names)

['Bacterial Red disease', 'Bacterial diseases - Aeromoniasis', 'Bacterial gill disease', 'Fungal diseases Saprolegniasis', 'Healthy Fish', 'Parasitic diseases', 'Viral diseases White tail disease']


In [16]:
print(len(train_loader))

88


In [17]:
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

print("Classes:", class_names)
print("Number of classes:", len(class_names))

Train batches: 88
Val batches: 22
Test batches: 44
Classes: ['Bacterial Red disease', 'Bacterial diseases - Aeromoniasis', 'Bacterial gill disease', 'Fungal diseases Saprolegniasis', 'Healthy Fish', 'Parasitic diseases', 'Viral diseases White tail disease']
Number of classes: 7


In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [22]:
import torch.nn as nn
from torchvision import models

model = models.densenet121(weights="DEFAULT")

in_features = model.classifier.in_features

model.classifier = nn.Linear(
    in_features,
    len(class_names)
)

model = model.to(device)

print("DenseNet121 Loaded Successfully")

DenseNet121 Loaded Successfully


In [25]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-5,
    weight_decay=1e-4
)

In [27]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3
)

In [29]:
EPOCHS = 30

best_acc = 0

train_losses = []
val_losses = []

train_accs = []
val_accs = []

In [30]:
print(next(model.parameters()).device)

cuda:0


In [31]:
for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # ======================
    # TRAINING
    # ======================

    model.train()

    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

    train_loss_epoch = train_loss / len(train_loader)
    train_acc_epoch = correct / total

    train_losses.append(train_loss_epoch)
    train_accs.append(train_acc_epoch)

    print(
        f"Train Loss: {train_loss_epoch:.4f}, "
        f"Train Acc: {train_acc_epoch:.4f}"
    )

    # ======================
    # VALIDATION
    # ======================

    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()

            total += labels.size(0)

    val_loss_epoch = val_loss / len(val_loader)
    val_acc_epoch = correct / total

    val_losses.append(val_loss_epoch)
    val_accs.append(val_acc_epoch)

    print(
        f"Val Loss: {val_loss_epoch:.4f}, "
        f"Val Acc: {val_acc_epoch:.4f}"
    )

    scheduler.step(val_acc_epoch)

    if val_acc_epoch > best_acc:

        best_acc = val_acc_epoch

        torch.save(
            model.state_dict(),
            "/kaggle/working/best_densenet121.pth"
        )

        print("✅ Best Model Saved")


Epoch 1/30
Train Loss: 1.6610, Train Acc: 0.4050
Val Loss: 1.3195, Val Acc: 0.6286
✅ Best Model Saved

Epoch 2/30
Train Loss: 1.0735, Train Acc: 0.7493
Val Loss: 0.9065, Val Acc: 0.7714
✅ Best Model Saved

Epoch 3/30
Train Loss: 0.7377, Train Acc: 0.8336
Val Loss: 0.6491, Val Acc: 0.8143
✅ Best Model Saved

Epoch 4/30
Train Loss: 0.5141, Train Acc: 0.8921
Val Loss: 0.4719, Val Acc: 0.8743
✅ Best Model Saved

Epoch 5/30
Train Loss: 0.3881, Train Acc: 0.9286
Val Loss: 0.3670, Val Acc: 0.9029
✅ Best Model Saved

Epoch 6/30
Train Loss: 0.2652, Train Acc: 0.9600
Val Loss: 0.3139, Val Acc: 0.9200
✅ Best Model Saved

Epoch 7/30
Train Loss: 0.2027, Train Acc: 0.9750
Val Loss: 0.2541, Val Acc: 0.9257
✅ Best Model Saved

Epoch 8/30
Train Loss: 0.1491, Train Acc: 0.9821
Val Loss: 0.2323, Val Acc: 0.9400
✅ Best Model Saved

Epoch 9/30
Train Loss: 0.1235, Train Acc: 0.9800
Val Loss: 0.2113, Val Acc: 0.9514
✅ Best Model Saved

Epoch 10/30
Train Loss: 0.1031, Train Acc: 0.9879
Val Loss: 0.2000, Val 

In [32]:
model.load_state_dict(
    torch.load(
        "/kaggle/working/best_densenet121.pth"
    )
)

model.eval()

print("Best Model Loaded")

Best Model Loaded


In [33]:
correct = 0
total = 0

all_test_labels = []
all_test_preds = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

        all_test_labels.extend(labels.cpu().numpy())
        all_test_preds.extend(preds.cpu().numpy())

test_acc = 100 * correct / total

print(f"\nTest Accuracy: {test_acc:.2f}%")


Test Accuracy: 99.29%


In [34]:
from sklearn.metrics import classification_report

print(
    classification_report(
        all_test_labels,
        all_test_preds,
        target_names=class_names
    )
)

                                   precision    recall  f1-score   support

            Bacterial Red disease       0.99      1.00      1.00       100
Bacterial diseases - Aeromoniasis       1.00      0.98      0.99       100
           Bacterial gill disease       0.97      0.99      0.98       100
   Fungal diseases Saprolegniasis       0.99      0.99      0.99       100
                     Healthy Fish       1.00      1.00      1.00       100
               Parasitic diseases       1.00      1.00      1.00       100
Viral diseases White tail disease       1.00      0.99      0.99       100

                         accuracy                           0.99       700
                        macro avg       0.99      0.99      0.99       700
                     weighted avg       0.99      0.99      0.99       700

